# Hands-on 6 — Patch embeddings and text search with Major TOM

**Chapter 5 · Patch embeddings and open models**

So far every embedding described one pixel. Major TOM's embeddings (ESA Φ-lab and CloudFerro) describe **whole image patches** of 384 × 384 Sentinel-2 pixels, about 3.8 km on a side, and they come from *general-purpose* vision models rather than EO-specific ones. One of those models, SigLIP, was trained on images **and their captions**, so its image embeddings live in the same space as text embeddings. That gives you something no pixel product offers: search satellite imagery with a sentence.

In this notebook you will:

1. load and understand a Major TOM embedding table (GeoParquet, one row per patch),
2. do image-to-image search: "patches like this one",
3. do text-to-image search: "a lake", "agricultural fields", "a burned forest",
4. classify every patch zero-shot with a handful of text prompts,
5. see where patch granularity and web-photo training let you down,
6. learn how to pull the same data for a whole country from Hugging Face.

Run time about 2 minutes. Everything needed is shipped with the course (3.5 MB subset for Attica plus precomputed text embeddings). An optional cell downloads the 3.5 GB SigLIP model so you can type your own queries.

In [ ]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import duckdb
import folium

import eoemb

## 1. The embedding table

Major TOM (Terrestrial Observation Metaset) is first of all a **grid**: the Earth divided into cells about 10 km across, named like `424U_208R` (row 424 north of the equator, column 208 to the right of the cell's row origin). The Core datasets hold Sentinel-1 and Sentinel-2 imagery per cell; the embedding datasets hold one vector per 384 × 384 patch cut from those images, nine patches per cell with 10 % overlap.

The course ships the Attica rows of `Major-TOM/Core-S2RGB-SigLIP` as a Parquet file.

In [ ]:
con = duckdb.connect()
src = eoemb.DATA_DIR / "majortom_s2rgb_siglip_attica.parquet"
print(con.execute(f"DESCRIBE SELECT * FROM '{src}'").df()[["column_name", "column_type"]].to_string(index=False))

df = con.execute(f"""
    SELECT unique_id, grid_cell, timestamp, product_id, centre_lat, centre_lon, utm_crs, utm_footprint, geometry, embedding
    FROM '{src}'""").df()
print(f"\n{len(df)} patches, {df.grid_cell.nunique()} grid cells, {df.product_id.nunique()} Sentinel-2 products, "
      f"dates {df.timestamp.min()[:8]} – {df.timestamp.max()[:8]}")
df[["grid_cell", "timestamp", "centre_lat", "centre_lon"]].head()

Points to notice:

- `embedding` is a list of 1152 floats: the output of the SigLIP-SO400M image encoder. Not unit length, so we normalise before computing cosine similarity.
- `geometry` is the patch footprint in WGS 84 (as WKB), `utm_footprint` the same in the local UTM zone.
- `timestamp` and `product_id` tell you *which* Sentinel-2 acquisition the patch came from. One cell may have several dates in the full dataset; here it is one date per cell, and the dates span 2017–2023. **Each embedding describes one image on one day**, not a year. That is the other big difference from AlphaEarth and TESSERA.

In [ ]:
E = np.stack(df.embedding.values).astype(np.float32)
E /= np.linalg.norm(E, axis=1, keepdims=True)
patches = gpd.GeoDataFrame(df.drop(columns=["embedding", "geometry"]), geometry=gpd.GeoSeries.from_wkb(df.geometry.apply(bytes)), crs="EPSG:4326")
print("embeddings", E.shape, "| footprint of the first patch:", patches.geometry.iloc[0].bounds)

Where are the patches? Coloured by their first three principal components, on top of a Sentinel-2 cloudless mosaic.

In [ ]:
from sklearn.decomposition import PCA

pc = PCA(3, random_state=0).fit_transform(E)
lo, hi = np.percentile(pc, [2, 98], axis=0)
colors = ["#%02x%02x%02x" % tuple(int(v * 255) for v in c) for c in np.clip((pc - lo) / (hi - lo), 0, 1)]
patches["color"] = colors

S2_TILES = "https://tiles.maps.eox.at/wmts/1.0.0/s2cloudless-2023_3857/default/g/{z}/{y}/{x}.jpg"
S2_ATTR = "Sentinel-2 cloudless by EOX IT Services GmbH (CC BY-NC-SA 4.0), contains modified Copernicus Sentinel data 2023"

def base_map(center=(38.1, 23.8), zoom=8):
    m = folium.Map(location=center, zoom_start=zoom, tiles=None)
    folium.TileLayer(S2_TILES, attr=S2_ATTR, name="Sentinel-2 cloudless 2023").add_to(m)
    return m

m = base_map()
folium.GeoJson(patches[["grid_cell", "timestamp", "color", "geometry"]],
               style_function=lambda f: {"color": f["properties"]["color"], "fillColor": f["properties"]["color"], "weight": 0.5, "fillOpacity": 0.55},
               tooltip=folium.GeoJsonTooltip(["grid_cell", "timestamp"])).add_to(m)
m

Look before you trust. The colours vary more **between grid rows** than between landscapes: whole rows of patches share a hue. Dates are mixed within each row (14 acquisition dates per row in this subset), so it is not simply "one date per row". Something else that is constant along a row, orbit geometry, processing baseline, or how the RGB patches were scaled, leaves a mark on the general-purpose SigLIP embedding that is as strong as the landscape itself. Exercise E4 asks you to investigate. The practical lesson is general: **always plot an embedding product over space before you use it**; artefacts of the production pipeline show up as geometry that has nothing to do with the ground.

## 2. Image-to-image: patches like this one

Same idea as Hands-on 3, one level up: the query is a whole patch. Take the patch that contains Piraeus and the port.

In [ ]:
def patch_at(lon, lat):
    hit = patches[patches.contains(gpd.points_from_xy([lon], [lat])[0])]
    if hit.empty:
        raise LookupError("no patch here")
    return hit.index[0]

def similar_patches(i, k=8):
    sims = E @ E[i]
    order = np.argsort(-sims)
    order = order[order != i][:k]
    return patches.iloc[order].assign(similarity=sims[order])

def show_patches(query_idx, hits, title):
    m = base_map(center=(patches.geometry.iloc[query_idx].centroid.y, patches.geometry.iloc[query_idx].centroid.x), zoom=9)
    folium.GeoJson(patches.iloc[[query_idx]].geometry, style_function=lambda f: {"color": "yellow", "weight": 3, "fillOpacity": 0}).add_to(m)
    folium.GeoJson(hits[["similarity", "grid_cell", "timestamp", "geometry"]],
                   style_function=lambda f: {"color": "red", "weight": 2, "fillOpacity": 0.1},
                   tooltip=folium.GeoJsonTooltip(["similarity", "grid_cell", "timestamp"])).add_to(m)
    folium.map.Marker(m.location, icon=folium.DivIcon(html=f'<div style="font-size:14pt;color:white;background:rgba(0,0,0,.5);padding:2px">{title}</div>')).add_to(m)
    return m

q = patch_at(23.62, 37.94)
hits = similar_patches(q)
print(hits[["grid_cell", "timestamp", "similarity"]].to_string())
show_patches(q, hits, "yellow = query (Piraeus), red = most similar")

The nearest patches are other dense urban and industrial waterfronts. Similarities are high (0.8–0.9) because SigLIP embeddings of satellite images all live in a fairly narrow cone: the model was trained on web photos and satellite images are "similar to each other" from its point of view. Rankings are what matter, not absolute values.

## 3. Text-to-image search

SigLIP is a vision-*language* model: its text tower maps a sentence into the same space as the image tower. The course ships text embeddings for 50 landscape descriptions (each averaged over three phrasings) computed with the same checkpoint Major TOM used, so you do not need the 3.5 GB model to try this.

In [ ]:
prompts = np.load(eoemb.DATA_DIR / "siglip_text_prompts.npz")
subjects = list(prompts["subjects"]); T = prompts["subject_embeddings"]
print(len(subjects), "subjects, e.g.", subjects[:12], "...")

def text_search(subject, k=6):
    t = T[subjects.index(subject)]
    sims = E @ t
    order = np.argsort(-sims)[:k]
    return patches.iloc[order].assign(similarity=sims[order])

for s in ["a lake", "agricultural fields", "a small island and open sea"]:
    h = text_search(s)
    print(f"\n'{s}':  top similarities {np.round(h.similarity.values, 3)}")
    print(h[["grid_cell", "centre_lat", "centre_lon"]].to_string(index=False))

In [ ]:
h = text_search("a lake")
m = base_map(zoom=8)
folium.GeoJson(h[["similarity", "grid_cell", "geometry"]], style_function=lambda f: {"color": "cyan", "weight": 3, "fillOpacity": 0.1},
               tooltip=folium.GeoJsonTooltip(["similarity", "grid_cell"])).add_to(m)
m

Lake Yliki and Lake Paralimni in Boeotia, the Marathon reservoir: found from the word "lake", with no training data whatsoever. Similarities are low in absolute terms (0.15–0.25 is normal for SigLIP text–image pairs); again, only the ranking counts.

Try a few more, and note the failures.

In [ ]:
places = {"Athens": (23.73, 37.98), "Piraeus": (23.64, 37.94), "Lake Yliki": (23.28, 38.40), "Chalkida": (23.60, 38.46),
          "Athens airport": (23.94, 37.94), "Marathon": (23.96, 38.15)}
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, s in zip(axes.ravel(), ["a pine forest on a mountain", "a burned forest", "a harbor with ships", "an airport runway", "a dense city center", "a coastline with beaches"]):
    h = text_search(s, k=10)
    patches.plot(ax=ax, color=patches.color, edgecolor="white", linewidth=0.2, alpha=0.35)   # PCA colours as faint context
    h.plot(ax=ax, color="red", edgecolor="darkred", alpha=0.9)
    for name, (lon, lat) in places.items():
        ax.plot(lon, lat, "k+", ms=8); ax.annotate(name, (lon, lat), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_title(f"'{s}' — top 10 patches"); ax.set_axis_off()
plt.tight_layout()

"Forest", "fields", "coastline" and "island" work. "Dense city center" lands on the dense northern suburbs rather than on central Athens. **"Harbor" and "airport" do not work**: the top hits are wrong or scattered. Two reasons, both worth remembering:

1. **Granularity.** A patch is 3.8 km across. A port or a runway occupies a few percent of it; the embedding describes the whole patch, which is mostly city, sea or farmland. Patch embeddings answer "what kind of landscape is this?", not "is there an X here?". For objects you need pixel embeddings (Hands-on 3) or smaller patches.
2. **Domain.** SigLIP learned "harbor" from holiday photos taken from the quay, not from 10 m nadir imagery. EO-specific vision-language models (RemoteCLIP, SkyCLIP, and successors) narrow this gap; Major TOM's other embedding sets (DINOv2, SSL4EO) do not have a text tower at all.

## 4. Zero-shot classification of every patch

If text queries rank patches, a set of text queries can *classify* them: each patch gets the label whose text embedding it is closest to. No labels, no training, one line of NumPy.

In [ ]:
classes = ["a dense city center", "a suburb with houses", "agricultural fields", "a pine forest on a mountain", "a bare rocky mountain", "open sea", "a coastline with beaches", "a lake"]
Tc = T[[subjects.index(c) for c in classes]]
scores = E @ Tc.T                                  # (patches, classes)
patches["zero_shot"] = np.array(classes)[scores.argmax(axis=1)]
patches["margin"] = np.sort(scores, axis=1)[:, -1] - np.sort(scores, axis=1)[:, -2]
print(patches.zero_shot.value_counts())

palette = dict(zip(classes, ["#7b1fa2", "#e91e63", "#fbc02d", "#1b5e20", "#8d6e63", "#0d47a1", "#00acc1", "#42a5f5"]))
m = base_map(zoom=8)
folium.GeoJson(patches[["zero_shot", "margin", "grid_cell", "geometry"]],
               style_function=lambda f: {"color": palette[f["properties"]["zero_shot"]], "fillColor": palette[f["properties"]["zero_shot"]], "weight": 0.3, "fillOpacity": 0.6},
               tooltip=folium.GeoJsonTooltip(["zero_shot", "margin", "grid_cell"])).add_to(m)
legend = "".join(f'<div><span style="background:{c};display:inline-block;width:14px;height:14px;margin-right:6px"></span>{n}</div>' for n, c in palette.items())
m.get_root().html.add_child(folium.Element(f'<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:8px;font-size:12px">{legend}</div>'))
m

A landscape map of Attica from eight sentences. It is coarse and it makes mistakes (hover to see the `margin`: small margins mean the model hesitated), but as a **first look at an unfamiliar region**, or to pre-select patches for a human to label, it is remarkable for the effort involved.

## 5. Type your own query (optional, 3.5 GB download)

The text embeddings above were precomputed. To search with your own words you need the SigLIP text tower. Set `RUN_MODEL = True` to download `google/siglip-so400m-patch14-384` from Hugging Face (once, ~3.5 GB, cached) and encode any sentence on the CPU in well under a second.

In [ ]:
RUN_MODEL = False

if RUN_MODEL:
    import torch
    from transformers import AutoTokenizer, AutoModel

    name = "google/siglip-so400m-patch14-384"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).eval()

    def embed_text(sentences):
        with torch.no_grad():
            inp = tok(sentences, padding="max_length", max_length=64, truncation=True, return_tensors="pt")
            out = model.get_text_features(**inp)
            t = (out.pooler_output if hasattr(out, "pooler_output") else out).numpy()
        return t / np.linalg.norm(t, axis=1, keepdims=True)

    my = embed_text(["a satellite image of greenhouses and plastic-covered fields."])[0]
    order = np.argsort(-(E @ my))[:6]
    print(patches.iloc[order][["grid_cell", "centre_lat", "centre_lon"]])

## 6. Getting Major TOM embeddings for your own region

The full `Core-S2RGB-SigLIP` set is 46 Parquet files of about 1 GB on Hugging Face, **sorted by latitude**. So you never need all of it: find the files whose latitude range overlaps your region (their footers say so), download those, and filter with DuckDB. For Greece (34.7–41.8° N) that is four files. The code below does it for the two files covering 36.5–41.1° N (Attica to Macedonia); set `RUN_FULL = True` to execute (about 2 GB, cached by `huggingface_hub`).

In [ ]:
RUN_FULL = False

if RUN_FULL:
    from huggingface_hub import hf_hub_download
    files = ["embeddings/part_03001-03100.parquet", "embeddings/part_03101-03200.parquet"]
    paths = [hf_hub_download("Major-TOM/Core-S2RGB-SigLIP", f, repo_type="dataset") for f in files]
    lst = ",".join(f"'{p}'" for p in paths)
    con.execute(f"""COPY (SELECT * FROM read_parquet([{lst}])
                     WHERE centre_lat BETWEEN 36.5 AND 41.1 AND centre_lon BETWEEN 19.3 AND 28.3)
                     TO '{eoemb.DATA_DIR / "cache" / "majortom_siglip_greece_north.parquet"}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    print(con.execute(f"SELECT count(*) FROM '{eoemb.DATA_DIR / 'cache' / 'majortom_siglip_greece_north.parquet'}'").fetchone())

To find the right files for *any* latitude without downloading them, read only the Parquet metadata over HTTP:

```python
con.execute("INSTALL httpfs; LOAD httpfs;")
url = "hf://datasets/Major-TOM/Core-S2RGB-SigLIP/embeddings/part_03001-03100.parquet"
con.execute(f"SELECT min(stats_min), max(stats_max) FROM parquet_metadata('{url}') WHERE path_in_schema = 'centre_lat'").fetchone()
```

The same recipe works for the sibling datasets `Core-S2L1C-SSL4EO` (multispectral, EO-specific model), `Core-S2RGB-DINOv2`, and `Core-S1RTC-SSL4EO` (radar). Same grid, same schema, different embedding spaces.

---
## Exercises

**E1.** Search for `"a quarry or open-pit mine"` and `"an industrial area"` with `text_search`. Check the top hits on the map. Which one works, and why (think granularity)?

**E2.** Take the "fields" patch containing (23.08, 38.37) in Boeotia as an image query. Compare its top-8 with the text search for `"agricultural fields"`. How many patches do the two lists share?

**E3.** The file `siglip_text_prompts.npz` also contains `prompt_embeddings`: the three individual phrasings per subject ("a satellite image of {}", "an aerial photo of {}", "{}"). For `"a lake"`, run the search with each phrasing separately. How stable is the top-5? This is why we average phrasings ("prompt ensembling").


**E4.** Investigate the row banding of section 1: group patches by `grid_row_u` and by the processing baseline in `product_id` (the `N02xx` part), average the embeddings per group, and see which grouping explains more of the variance. Is the artefact in the model, or in how the input images were prepared?